# Autoregressive Video Generation

Why AR for world models:

1. **Real-time generation** -- produce frames as they're needed, don't wait for the whole video. A diffusion model must denoise all frames jointly before outputting anything; AR outputs frame 0 immediately.
2. **Interactive** -- condition on user actions mid-generation. At frame N, the user clicks "turn left" and frame N+1 responds. Impossible with bidirectional diffusion that commits to all frames upfront.
3. **Linear complexity for long videos** -- generate frame N given frames 0..N-1. Diffusion processes all frames jointly with quadratic attention over the full sequence. For a 10-minute video at 24 FPS (14,400 frames), quadratic is a non-starter.
4. **Natural fit for "world models"** -- predict next state given current state + action, like a game engine. The AR formulation $p(s_{t+1} | s_t, a_t)$ maps directly to world model semantics.

**The catch: error accumulation.** Each predicted frame feeds into the next, so mistakes compound. A small drift in frame 10 becomes a large artifact by frame 100. This is THE core challenge and the reason diffusion still dominates quality benchmarks.

**Time:** ~2-3 hrs. Training cells need GPU.

## Approach Comparison

```
DIFFUSION (bidirectional):
  All frames generated simultaneously
  [F0, F1, F2, ..., FN] <- denoise together
  + High visual quality (all frames see each other)
  - Must decide frame count upfront
  - Can't stream or interact mid-generation

AUTOREGRESSIVE (causal):
  Frames generated one at a time
  F0 -> F1 -> F2 -> ... -> FN
  + Streaming, interactive, variable length
  - Error accumulation
  - Sequential bottleneck

HYBRID A2D (Gen-4.5):
  AR for temporal/semantic structure
  Diffusion for visual fidelity per frame
  Best of both: stream + high quality
```

The key insight: AR and diffusion are not mutually exclusive. You can use AR to decide *what happens* (semantics, motion, scene transitions) and diffusion to decide *how it looks* (texture, fine detail, coherence within a frame). This is exactly what the A2D (Autoregressive-to-Diffusion) architecture does.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# --- Causal vs Bidirectional Attention Masks ---

def make_causal_mask(n: int) -> torch.Tensor:
    """Causal mask: frame i can only attend to frames 0..i."""
    return torch.tril(torch.ones(n, n))

def make_full_mask(n: int) -> torch.Tensor:
    """Full attention: every frame sees every frame."""
    return torch.ones(n, n)

n_frames = 8
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

causal = make_causal_mask(n_frames)
axes[0].imshow(causal, cmap="Blues")
axes[0].set_title("Causal Attention (AR)\nFrame i sees frames 0..i")
axes[0].set_xlabel("Key frame")
axes[0].set_ylabel("Query frame")
for i in range(n_frames):
    for j in range(n_frames):
        axes[0].text(j, i, f"{'1' if causal[i,j] else '0'}", ha="center", va="center", fontsize=10)

full = make_full_mask(n_frames)
axes[1].imshow(full, cmap="Blues")
axes[1].set_title("Bidirectional Attention (Diffusion)\nEvery frame sees every frame")
axes[1].set_xlabel("Key frame")
axes[1].set_ylabel("Query frame")
for i in range(n_frames):
    for j in range(n_frames):
        axes[1].text(j, i, "1", ha="center", va="center", fontsize=10)

plt.tight_layout()
plt.show()

# Key insight: causal mask is what makes AR work -- information only flows forward in time
print(f"Causal: each frame attends to avg {causal.sum(1).mean():.1f} frames")
print(f"Full: each frame attends to {full.sum(1).mean():.1f} frames")
print(f"Causal saves {(1 - causal.sum()/full.sum())*100:.0f}% of attention computation")

In [ ]:
# --- Bouncing Ball Dataset ---
# A simple synthetic video dataset: a ball bouncing inside a 32x32 frame.
# Physics are deterministic given initial conditions, so a perfect model should
# predict future frames exactly from seed frames.

def generate_bouncing_ball(n_frames: int = 16, size: int = 32, ball_r: int = 3) -> torch.Tensor:
    """Generate a simple bouncing ball video."""
    frames = np.zeros((n_frames, 1, size, size), dtype=np.float32)
    # Random initial position and velocity
    x, y = np.random.uniform(ball_r + 1, size - ball_r - 1, 2)
    vx, vy = np.random.uniform(-2, 2, 2)

    for t in range(n_frames):
        # Draw ball
        yy, xx = np.ogrid[:size, :size]
        mask = ((xx - x) ** 2 + (yy - y) ** 2) <= ball_r ** 2
        frames[t, 0] = mask.astype(np.float32)
        # Update position with bouncing
        x += vx
        y += vy
        if x <= ball_r or x >= size - ball_r:
            vx *= -1
            x = np.clip(x, ball_r, size - ball_r)
        if y <= ball_r or y >= size - ball_r:
            vy *= -1
            y = np.clip(y, ball_r, size - ball_r)

    return torch.from_numpy(frames)

# Generate dataset
np.random.seed(42)
n_samples = 2000
dataset = torch.stack([generate_bouncing_ball() for _ in tqdm(range(n_samples), desc="Generating data")])
print(f"Dataset shape: {dataset.shape}")  # [2000, 16, 1, 32, 32]

# Visualize one sample
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
sample = dataset[0]
for i in range(8):
    axes[i].imshow(sample[i * 2, 0], cmap="gray")
    axes[i].set_title(f"t={i * 2}")
    axes[i].axis("off")
plt.suptitle("Bouncing Ball Sequence (every 2nd frame)")
plt.tight_layout()
plt.show()

In [ ]:
# --- AR Video Transformer ---
# Patchify each frame, add spatial + temporal position embeddings,
# apply causal attention across time, decode back to pixels.
# GPU: ~1-2 GB VRAM

class PatchEmbed(nn.Module):
    """Convert frame to patch tokens."""
    def __init__(self, img_size: int = 32, patch_size: int = 4, in_channels: int = 1, embed_dim: int = 128):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        self.n_patches = (img_size // patch_size) ** 2

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x).flatten(2).transpose(1, 2)  # [B, n_patches, embed_dim]


class ARVideoTransformer(nn.Module):
    """Autoregressive video transformer: predict next frame from previous frames."""
    def __init__(
        self,
        img_size: int = 32,
        patch_size: int = 4,
        embed_dim: int = 128,
        n_heads: int = 4,
        n_layers: int = 4,
        max_frames: int = 16,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, 1, embed_dim)
        n_patches = (img_size // patch_size) ** 2

        # Positional embeddings: spatial + temporal
        self.spatial_pos = nn.Parameter(torch.randn(1, n_patches, embed_dim) * 0.02)
        self.temporal_pos = nn.Parameter(torch.randn(1, max_frames, 1, embed_dim) * 0.02)

        # Transformer with causal masking
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Decode back to pixels
        self.head = nn.Linear(embed_dim, patch_size * patch_size)
        self.patch_size = patch_size
        self.img_size = img_size
        self.n_patches = n_patches
        self.embed_dim = embed_dim

    def _make_causal_mask(self, T: int, device: torch.device) -> torch.Tensor:
        """Build causal mask: tokens from frame t attend to frames 0..t only."""
        total_tokens = T * self.n_patches
        mask = torch.full((total_tokens, total_tokens), float("-inf"), device=device)
        for t in range(T):
            start = t * self.n_patches
            end = (t + 1) * self.n_patches
            # Frame t can attend to all tokens from frames 0..t
            mask[start:end, :end] = 0.0
        return mask

    def forward(self, frames: torch.Tensor) -> torch.Tensor:
        """frames: [B, T, 1, H, W] -> predicted frames [B, T, 1, H, W]."""
        B, T, C, H, W = frames.shape

        # Patchify each frame and add positional embeddings
        tokens = []
        for t in range(T):
            patch_tokens = self.patch_embed(frames[:, t])  # [B, n_patches, embed_dim]
            patch_tokens = patch_tokens + self.spatial_pos + self.temporal_pos[:, t : t + 1]
            tokens.append(patch_tokens)
        tokens = torch.cat(tokens, dim=1)  # [B, T*n_patches, embed_dim]

        # Causal mask
        mask = self._make_causal_mask(T, frames.device)

        out = self.transformer(tokens, mask=mask)

        # Predict pixels
        predictions = self.head(out)  # [B, T*n_patches, patch_size^2]

        # Reshape to frames
        p = self.patch_size
        n_side = self.img_size // p
        pred_frames = predictions.reshape(B, T, self.n_patches, p, p)
        pred_frames = pred_frames.reshape(B, T, n_side, n_side, p, p)
        pred_frames = pred_frames.permute(0, 1, 2, 4, 3, 5).reshape(B, T, 1, H, W)

        return pred_frames


model = ARVideoTransformer().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, 8, 1, 32, 32, device=device)
    out = model(dummy)
    print(f"Input: {dummy.shape} -> Output: {out.shape}")

In [ ]:
# --- Training: Predict Next Frame from Context ---
# Teacher forcing: input frames 0..T-2, predict frames 1..T-1.
# GPU: ~2-3 GB VRAM with batch_size=32

train_data = TensorDataset(dataset[:1600])
val_data = TensorDataset(dataset[1600:])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

train_losses = []
val_losses = []

for epoch in tqdm(range(30), desc="Training"):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    for (batch,) in train_loader:
        batch = batch.to(device)
        # Input: frames 0..T-2, Target: frames 1..T-1
        input_frames = batch[:, :-1]
        target_frames = batch[:, 1:]

        pred = model(input_frames)
        loss = F.mse_loss(pred, target_frames)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()
    avg_train = epoch_loss / len(train_loader)
    train_losses.append(avg_train)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (batch,) in val_loader:
            batch = batch.to(device)
            pred = model(batch[:, :-1])
            val_loss += F.mse_loss(pred, batch[:, 1:]).item()
    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: train_loss={avg_train:.6f}, val_loss={avg_val:.6f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, label="Train")
ax.plot(val_losses, label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("AR Video Transformer Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Autoregressive Generation ---
# Feed predictions back as input. This is where error accumulation happens:
# during training, the model always sees ground-truth context (teacher forcing).
# At inference, it sees its own (imperfect) predictions.

@torch.no_grad()
def generate_ar(
    model: nn.Module,
    seed_frames: torch.Tensor,
    n_generate: int = 12,
) -> list[torch.Tensor]:
    """Generate frames autoregressively: each prediction feeds back as input."""
    model.eval()
    frames = seed_frames.clone()  # [1, n_seed, 1, H, W]

    all_frames = [frames[0, i, 0].cpu() for i in range(frames.shape[1])]

    for step in range(n_generate):
        pred = model(frames)
        next_frame = pred[:, -1:].clamp(0, 1)  # Take prediction for last position
        frames = torch.cat([frames, next_frame], dim=1)
        all_frames.append(next_frame[0, 0, 0].cpu())

    return all_frames


# Generate from 4 seed frames
test_idx = 1800
seed = dataset[test_idx : test_idx + 1, :4].to(device)
generated = generate_ar(model, seed, n_generate=12)

# Visualize: seed frames (green border) + generated (red border)
fig, axes = plt.subplots(1, 16, figsize=(24, 2))
for i, frame in enumerate(generated):
    axes[i].imshow(frame, cmap="gray", vmin=0, vmax=1)
    color = "green" if i < 4 else "red"
    for spine in axes[i].spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)
    axes[i].set_title(f"t={i}", fontsize=8)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
plt.suptitle("AR Generation: Green=seed, Red=generated")
plt.tight_layout()
plt.show()

# Also show ground truth for comparison
fig, axes = plt.subplots(1, 16, figsize=(24, 2))
gt = dataset[test_idx]
for i in range(16):
    axes[i].imshow(gt[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[i].set_title(f"t={i}", fontsize=8)
    axes[i].set_xticks([])
    axes[i].set_yticks([])
plt.suptitle("Ground Truth")
plt.tight_layout()
plt.show()

In [ ]:
## CausVid: Bidirectional to Causal Distillation (CVPR 2025, arXiv 2412.07772)

Extends DMD-style distillation to video with a critical twist: converting bidirectional attention to causal.

**Setup:**
- **Teacher:** Trained bidirectional video diffusion model (all frames see all frames). High quality but can't stream.
- **Student:** Same architecture but with causal attention mask (frame t only sees frames 0..t-1). Can stream.

**Key innovations:**
1. **Asymmetric distillation:** Teacher uses full attention + 50 steps. Student uses causal attention + 4 steps. Two axes of compression simultaneously.
2. **DMD-style distribution matching adapted for video:** The "fake score" is trained on the student's video outputs, capturing temporal artifacts.
3. **Progressive training:** Start with many student steps, gradually reduce. The student first learns temporal consistency, then learns to maintain it with fewer steps.

**Result:** 50-step bidirectional -> 4-step causal. 1.3s initial latency, then streaming at 9.4 FPS.

**Why this matters for AR video generation:** It's the bridge from diffusion (quality) to AR (streaming). You get the quality of a model that was trained with full bidirectional attention, but deployed with causal attention for real-time use.

In [ ]:
# --- Attention Pattern Visualization ---
# What temporal patterns does the model learn? Does it attend mostly to recent
# frames (Markov-like) or distant ones?

model.eval()
sample = dataset[test_idx : test_idx + 1, :8].to(device)

# Get token representations and compute attention
with torch.no_grad():
    B, T, C, H, W = sample.shape
    tokens = []
    for t in range(T):
        patch_tokens = model.patch_embed(sample[:, t])
        patch_tokens = patch_tokens + model.spatial_pos + model.temporal_pos[:, t : t + 1]
        tokens.append(patch_tokens)
    tokens_cat = torch.cat(tokens, dim=1)

    # Compute attention weights from first transformer layer
    first_layer = model.transformer.layers[0]
    # nn.TransformerEncoderLayer wraps nn.MultiheadAttention
    # Call self_attn directly with need_weights=True
    q = k = v = first_layer.norm1(tokens_cat)  # Pre-norm
    attn_out, attn_weights = first_layer.self_attn(
        q, k, v, need_weights=True, average_attn_weights=True
    )  # attn_weights: [B, T*n_patches, T*n_patches]

    # Average over spatial patches to get frame-level attention
    n_patches = model.n_patches
    frame_attn = torch.zeros(T, T)
    for qi in range(T):
        for ki in range(T):
            frame_attn[qi, ki] = attn_weights[
                0,
                qi * n_patches : (qi + 1) * n_patches,
                ki * n_patches : (ki + 1) * n_patches,
            ].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frame-level attention heatmap
im = axes[0].imshow(frame_attn.cpu(), cmap="hot")
plt.colorbar(im, ax=axes[0], label="Attention weight")
axes[0].set_xlabel("Key frame")
axes[0].set_ylabel("Query frame")
axes[0].set_title("Frame-to-Frame Attention Weights\n(which past frames matter most?)")

# Per-query attention distribution
for qi in range(T):
    axes[1].plot(range(qi + 1), frame_attn[qi, : qi + 1].cpu(), "o-", label=f"Frame {qi}", alpha=0.7)
axes[1].set_xlabel("Key frame")
axes[1].set_ylabel("Attention weight")
axes[1].set_title("Attention Distribution per Query Frame")
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Analysis: does the model learn recency bias?
print("\nRecency analysis (how much attention goes to the most recent frame):")
for qi in range(1, T):
    total_attn = frame_attn[qi, :qi + 1].sum()
    recent_attn = frame_attn[qi, qi] / total_attn
    print(f"  Frame {qi}: {recent_attn:.1%} attention on most recent (self), "
          f"{frame_attn[qi, qi-1] / total_attn:.1%} on frame {qi-1}")

## Notable AR Approaches in the Field

**A2D Architecture (Gen-4.5):**
- AR handles temporal structure and semantics (what happens next)
- Diffusion handles visual fidelity per frame (how it looks)
- Hybrid: streaming/interaction from AR + quality from diffusion
- Tech report demonstrates "world model" properties: the model learns physics, object permanence, and 3D consistency without explicit supervision

**GWM-1 (General World Model):**
- Fully autoregressive, frame-by-frame generation
- Real-time generation with action conditioning
- Like a neural game engine: take action -> see next frame
- Key enabler for interactive applications (games, simulations, embodied AI)

The strategic bet: video generation isn't just about making pretty videos. It's about building models that understand the world well enough to simulate it interactively. AR is the natural architecture for simulation.

## Key Research

**CausVid (CVPR 2025, arXiv 2412.07772):**
Convert a bidirectional diffusion model to causal AR via distillation. Train a student that only looks backward in time but matches the bidirectional teacher's output distribution. Result: 1.3s initial latency, then streaming at 9.4 FPS. The key insight is that you can distill the *information* from bidirectional attention into causal attention without fundamentally losing quality -- the causal student learns to predict what the future frames "would have told" the current frame.

**Causal Forcing:**
ODE-based distillation from an AR teacher, focusing on causal consistency. Rather than just matching output frames, it enforces that the denoising ODE trajectory for frame t is independent of frames t+1, t+2, etc. This is a stronger constraint than just causal attention masks -- it's a mathematical guarantee on the generation process.

**Error accumulation mitigations:**
- **Teacher forcing with scheduled sampling:** During training, gradually replace ground-truth context with model predictions. Bridges the train/inference gap.
- **Diffusion refinement per frame:** After AR generates a "draft" frame, run a few diffusion steps to clean it up before feeding it to the next step. Adds latency but breaks the error chain.
- **Multi-frame prediction:** Predict 4 frames at once instead of 1. Reduces the number of autoregressive steps (and thus error accumulation opportunities) by 4x, at the cost of losing frame-level interactivity.

## Checkpoint

Why use autoregressive approaches for world models? Because world models need to be:

1. **Interactive** -- respond to user actions in real time. Diffusion can't do this because it generates all frames jointly.
2. **Streamable** -- produce frames as they're needed. A user watching a generated video shouldn't wait 30 seconds for the whole thing to render.
3. **Open-ended** -- no fixed video length. An interactive world should run as long as the user wants.

Diffusion models generate all frames at once -- great for quality, bad for interaction. AR is the natural fit, but the quality gap vs diffusion is the unsolved problem. The A2D hybrid tries to get both: AR decides *what happens*, diffusion decides *how it looks*.

The error accumulation problem we demonstrated above is a toy version of the real challenge. On a bouncing ball, errors manifest as position drift. On real video, they manifest as texture degradation, identity loss, physics violations, and temporal incoherence. Solving this at scale is an active area of research across multiple teams in the field.

**Further reading:** CausVid (arXiv 2412.07772), Gen-4.5 Tech Report, GWM-1.